# Module 07 — Assignment: the LSTM cell, and backpropagation through it

Module 06's assignment built a layer that is deep in time and showed where it breaks: the gradient
through time is a power of one matrix. This one builds Hochreiter and Schmidhuber's answer. You
implement four load-bearing pieces — **one step of the LSTM cell** (three gates and an additive
update), the **forward pass over a sequence**, **backpropagation through time along both of its
paths** (with the 1997 cell-only gradient as an option), and the **gate biases** that decide how
long the cell remembers at the start — and the check cells grade you against finite differences and
the canonical `python/lstm.py` + `nanograd` answer key. The finale drops your layer into a GIVEN
harness that must reproduce the module's latch: with its gates open to the past and shut to new
input your LSTM holds a symbol through 79 steps of distractors; with the textbook forget bias of 1
it cannot; and trained with the error flowing back only along the cell, it still can.

Rules: don't peek at `lib/python/nanograd` or `python/lstm.py` until you've tried. The check cells
are your feedback. Fill every `# TODO`, run top to bottom.

In [ ]:
# Setup — run once. (GIVEN: do not modify.)
import os, sys, math
import numpy as np

def _find_repo():
    here = os.getcwd()
    cands = [here] + [os.path.abspath(os.path.join(here, *(['..'] * k))) for k in range(1, 6)]
    for base in cands:
        if os.path.isdir(os.path.join(base, 'lib', 'python', 'nanograd')):
            return base
    return here

REPO = _find_repo()
TOPIC = os.path.join(REPO, 'topics', '07-lstm')
sys.path.insert(0, os.path.join(REPO, 'lib', 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'python'))
sys.path.insert(0, os.path.join(TOPIC, 'tests'))
import nanograd as ng
import lstm as canon
from check_utils import (rel_error, eval_numerical_gradient_array, compare_to_canonical)
from nanograd.rng import Rng

def sigmoid(z):                                    # GIVEN
    return 1.0 / (1.0 + np.exp(-z))
print('setup ok — nanograd and check utils imported')

## Part 1 — one step of the cell

All four gate pre-activations come from one product, $z = x W_x + h^{\langle t-1\rangle} W_h + b$, laid
out side by side in four blocks of $n_h$ columns, in the order $[\,i \mid f \mid g \mid o\,]$:

$$i = \sigma(z_i),\quad f = \sigma(z_f),\quad g = \tanh(z_g),\quad o = \sigma(z_o),$$
$$c^{\langle t\rangle} = f \odot c^{\langle t-1\rangle} + i \odot g, \qquad h^{\langle t\rangle} = o \odot \tanh\big(c^{\langle t\rangle}\big).$$

Shapes: $x$ is $(m, n_{in})$, $h$ and $c$ are $(m, n_h)$, $W_x$ is $(n_{in}, 4n_h)$, $W_h$ is
$(n_h, 4n_h)$, $b$ is $(4n_h)$. Return $h$, $c$, and the activated gates side by side,
`gates = [i | f | g | o]` of shape $(m, 4n_h)$ — backpropagation will need them.

In [ ]:
def lstm_step(x, h_prev, c_prev, Wx, Wh, b):
    n = Wh.shape[0]
    h = c = gates = None
    ###########################################################################
    # TODO: the four pre-activations in one product; squash each block (sigmoid
    # for i, f, o; tanh for g); the new cell; the new state. Keep the activated
    # gates side by side in `gates`. ~7 lines.
    ###########################################################################

    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return h, c, gates

In [ ]:
# Check: one step equals nanograd's LSTM run for a single step from the same state and cell.
rng = np.random.default_rng(0)
lib = ng.LSTM(4, 6)
for p in lib.params():
    p[...] = rng.standard_normal(p.shape) * 0.5
x, h_prev, c_prev = rng.standard_normal((3, 4)), rng.standard_normal((3, 6)) * 0.5, rng.standard_normal((3, 6))
h, c, gates = lstm_step(x, h_prev, c_prev, lib.Wx, lib.Wh, lib.b)
H = lib.forward(x[None], h_prev, c_prev)
assert h is not None and h.shape == (3, 6) and c.shape == (3, 6), 'lstm_step should return (m, n_h) arrays'
assert gates.shape == (3, 24), 'gates should be (m, 4 n_h), blocks [i | f | g | o]'
for name, mine, theirs in (('h', h, H[0]), ('c', c, lib.C[0]), ('gates', gates, lib.G[0])):
    print(f'{name:5s} rel_error vs nanograd = {rel_error(mine, theirs):.1e}'); assert rel_error(mine, theirs) < 1e-12
compare_to_canonical([float(h.sum()), float(c.sum())], [float(H[0].sum()), float(lib.C[0].sum())], labels=['h_sum', 'c_sum'])

### Inline Question 1
(a) At 16 units and 8 inputs, how many parameters does Module 06's RNN layer have, and how many does
an LSTM layer have? Where does the factor come from? (b) Suppose $i = 0$ and $f = 1$ exactly. What is
$c^{\langle t\rangle}$, whatever $x^{\langle t\rangle}$ is? Can a recurrent unit $h = \tanh(w h + \ldots)$ do the
same for an arbitrary stored value — and if not, what can it do instead? (c) Why are the gates
sigmoids rather than, say, ReLUs? *Your answer:* ____

## Part 2 — the forward pass over a sequence

Time-major, as in Module 06: $X$ is $(T, m, n_{in})$. Starting from $h_0$ and $c_0$ (zeros if not
given), apply `lstm_step` at every step and keep **everything** backpropagation will need: the
states $H$ $(T, m, n_h)$, the cells $C$ $(T, m, n_h)$ and the gates $G$ $(T, m, 4n_h)$, as `self.H`,
`self.C`, `self.G`.

In [ ]:
class MyLSTM:
    def __init__(self, n_in, n_h):
        self.Wx, self.Wh, self.b = np.zeros((n_in, 4 * n_h)), np.zeros((n_h, 4 * n_h)), np.zeros(4 * n_h)  # GIVEN
        self.dWx, self.dWh, self.db = np.zeros_like(self.Wx), np.zeros_like(self.Wh), np.zeros_like(self.b)

    def forward(self, X, h0=None, c0=None):
        T, m, _ = X.shape                                          # GIVEN
        n = self.Wh.shape[0]
        self.X = X
        self.h0 = np.zeros((m, n)) if h0 is None else h0
        self.c0 = np.zeros((m, n)) if c0 is None else c0
        H = None
        ###########################################################################
        # TODO: run lstm_step over t = 0 .. T-1 from (self.h0, self.c0); keep every
        # state, cell and gate block in H, C, G; remember them as self.H, self.C,
        # self.G. ~8 lines.
        ###########################################################################

        ###########################################################################
        #                          END OF YOUR CODE
        ###########################################################################
        return H

    def backward(self, dH, cell_only=False):
        return my_lstm_bptt(self, dH, cell_only)                   # GIVEN: Part 3

    def params(self): return [self.Wx, self.Wh, self.b]
    def grads(self): return [self.dWx, self.dWh, self.db]

In [ ]:
# Check: identical to nanograd's LSTM on a sequence, with and without an initial state and cell.
rng = np.random.default_rng(1)
you, lib = MyLSTM(4, 6), ng.LSTM(4, 6)
for p, q in zip(you.params(), lib.params()):
    p[...] = q[...] = rng.standard_normal(p.shape) * 0.5
X = rng.standard_normal((9, 3, 4))
h0, c0 = rng.standard_normal((3, 6)) * 0.5, rng.standard_normal((3, 6))
for start in ((None, None), (h0, c0)):
    Hy, Hl = you.forward(X, *start), lib.forward(X, *start)
    assert Hy is not None and Hy.shape == (9, 3, 6), 'forward should return (T, m, n_h)'
    assert rel_error(Hy, Hl) < 1e-12 and rel_error(you.C, lib.C) < 1e-12, 'forward differs from nanograd'
print('every state and cell matches nanograd; the last cell:', np.round(you.C[-1, 0], 4))
compare_to_canonical([float(Hy.sum()), float(you.C[-1].sum())], [float(Hl.sum()), float(lib.C[-1].sum())],
                     labels=['H_sum', 'c_T'])

### Inline Question 2
Hold every forget gate at a constant $f$ and every input gate at 0. What fraction of $c^{\langle 0\rangle}$
is left after $k$ steps? For the customary forget bias $b_f = 1$ ($f = \sigma(1) \approx 0.73$), after how
many steps has the cell kept less than a hundredth of what it held? For $b_f = 5$? Which of the two
could carry a signal through Part 4's 79 distractors? *Your answer:* ____

## Part 3 — backpropagation through the cell

Walk time backwards, carrying **two** gradients from the future: through $h^{\langle t\rangle}$ (read by the
next step's gates) and through $c^{\langle t\rangle}$ (kept by the next step's cell):

$$dh^{\langle t\rangle} = dH^{\langle t\rangle} + dz^{\langle t+1\rangle} W_h^{\top}, \qquad
dc^{\langle t\rangle} = dc^{\langle t+1\rangle} \odot f^{\langle t+1\rangle} + dh^{\langle t\rangle} \odot o \odot \big(1 - \tanh^2 c^{\langle t\rangle}\big),$$

$$dz_i = dc\, g\, i(1-i),\quad dz_f = dc\, c^{\langle t-1\rangle} f(1-f),\quad dz_g = dc\, i\,(1-g^2),\quad dz_o = dh \tanh(c^{\langle t\rangle})\, o(1-o),$$

$$dW_x \mathrel{+}= x^{\langle t\rangle\top} dz,\qquad dW_h \mathrel{+}= h^{\langle t-1\rangle\top} dz,\qquad db \mathrel{+}= \textstyle\sum_{\text{rows}} dz,\qquad dX^{\langle t\rangle} = dz\, W_x^{\top},$$

with $h^{\langle -1\rangle}, c^{\langle -1\rangle} = h_0, c_0$; what is passed back from step 0 is $\partial L/\partial h_0$
and $\partial L/\partial c_0$. ($dz$ is the four blocks side by side, $[\,dz_i \mid dz_f \mid dz_g \mid dz_o\,]$.)

**The 1997 gradient.** With `cell_only=True`, nothing is passed back through $h$: the
$dz^{\langle t+1\rangle} W_h^{\top}$ term is dropped at every step (and $\partial L/\partial h_0 = 0$), while the cell
path $dc^{\langle t+1\rangle} f^{\langle t+1\rangle}$ still flows — Hochreiter and Schmidhuber's truncated
gradient, the constant error carousel alone.

Set `lstm.dWx`, `lstm.dWh`, `lstm.db`, `lstm.dh0`, `lstm.dc0`; return $dX$.

In [ ]:
def my_lstm_bptt(lstm, dH, cell_only=False):
    T, m, n = dH.shape
    dX = None
    ###########################################################################
    # TODO: zero the three weight gradients and dX; walk t = T-1 .. 0 carrying
    # dh_next and dc_next; at each step form dh and dc, the four gate gradients
    # dz, each weight's share, dX[t]; then pass dc * f one step back along the
    # cell and dz Wh^T along h (unless cell_only). Store dh0, dc0 = what is
    # passed back from step 0. ~20 lines.
    ###########################################################################

    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return dX

In [ ]:
# Check 1: finite differences, every input and parameter, with an initial state and cell.
rng = np.random.default_rng(2)
you = MyLSTM(3, 4)
for p in you.params():
    p[...] = rng.standard_normal(p.shape) * 0.6
X, dH = rng.standard_normal((7, 2, 3)), rng.standard_normal((7, 2, 4))
h0, c0 = rng.standard_normal((2, 4)) * 0.5, rng.standard_normal((2, 4)) * 0.5
f = lambda _: you.forward(X, h0, c0)
f(None); dX = you.backward(dH)
assert dX is not None, 'my_lstm_bptt should return dX'
err = rel_error(dX, eval_numerical_gradient_array(f, X, dH)); print(f'dX   rel_error = {err:.1e}'); assert err < 1e-6
for name, arr in (('h0', h0), ('c0', c0), ('Wx', you.Wx), ('Wh', you.Wh), ('b', you.b)):
    f(None); you.backward(dH); g = getattr(you, 'd' + name).copy()
    err = rel_error(g, eval_numerical_gradient_array(f, arr, dH))
    print(f'd{name:3s} rel_error = {err:.1e}'); assert err < 1e-6
lib = ng.LSTM(3, 4)
for p, q in zip(lib.params(), you.params()):
    p[...] = q
lib.forward(X, h0, c0); lib.backward(dH)
compare_to_canonical([float(you.dWx.sum()), float(you.dWh.sum()), float(you.db.sum()), float(you.dc0.sum())],
                     [float(lib.dWx.sum()), float(lib.dWh.sum()), float(lib.db.sum()), float(lib.dc0.sum())],
                     labels=['dWx', 'dWh', 'db', 'dc0'])

In [ ]:
# Check 2: the 1997 gradient is the exact gradient of a network whose gates read a FROZEN copy of
# h<t-1> -- so error can reach the past only along the cell. (GIVEN: the frozen forward.)
def frozen_forward(lstm, X, c0, H_read):
    n, c, out = lstm.Wh.shape[0], c0, []
    for t in range(X.shape[0]):
        z = X[t] @ lstm.Wx + H_read[t] @ lstm.Wh + lstm.b
        i, f, g, o = sigmoid(z[:, :n]), sigmoid(z[:, n:2*n]), np.tanh(z[:, 2*n:3*n]), sigmoid(z[:, 3*n:])
        c = f * c + i * g; out.append(o * np.tanh(c))
    return np.array(out)

H = you.forward(X, h0, c0)
H_read = np.concatenate([h0[None], H[:-1]])
you.backward(dH, cell_only=True)
for name, arr in (('Wx', you.Wx), ('Wh', you.Wh), ('b', you.b), ('c0', c0)):
    g = getattr(you, 'd' + name).copy()
    err = rel_error(g, eval_numerical_gradient_array(lambda _: frozen_forward(you, X, c0, H_read), arr, dH))
    print(f'cell-only d{name:3s} vs the frozen-h network: rel_error = {err:.1e}'); assert err < 1e-6
assert np.all(you.dh0 == 0), 'with cell_only, nothing reaches h0'
lib.forward(X, h0, c0); lib.backward(dH, cell_only=True)
compare_to_canonical([float(you.dWh.sum()), float(you.dc0.sum())], [float(lib.dWh.sum()), float(lib.dc0.sum())],
                     labels=['dWh (cell only)', 'dc0 (cell only)'])

### Inline Question 3
(a) With `cell_only=True` and an error only at the last step, write $\partial L/\partial c^{\langle t\rangle}$ in
terms of $\partial L/\partial c^{\langle T-1\rangle}$. Why can that gradient fade but never grow, however long
the sequence? (b) The full gradient adds the $h$ path, $dz^{\langle t+1\rangle} W_h^{\top}$. Which of Module 06's
two problems can that path bring back — and what does the module's toy measure about it (clipped
updates, largest gradient norm)? (c) Hochreiter and Schmidhuber also pointed out that the truncated
gradient needs no stored history of the sequence. Why not? *Your answer:* ____

## Part 4 — the gate biases

At the start of training every gate is roughly $\sigma$ of its bias, so the biases decide what the
cell does before it has learned anything. The forget gate's sets how long it remembers — about
$1/(1-\sigma(b_f))$ steps. The input gate's sets how much of every input it collects while it waits —
Hochreiter and Schmidhuber's remedy for *internal state drift* was "initially to bias the input gate
toward zero". Write the two helpers: one sets whole bias blocks of a layer, the other returns the
memory length a forget bias gives.

In [ ]:
def set_gate_biases(lstm, i, f):
    n = lstm.Wh.shape[0]                                           # GIVEN
    ###########################################################################
    # TODO: set the input gate's bias block to i and the forget gate's to f
    # (blocks [i | f | g | o], n_h each); leave g and o alone. 2 lines.
    ###########################################################################

    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################


def memory_steps(b_f):
    steps = None
    ###########################################################################
    # TODO: 1 / (1 - sigma(b_f)). One line.
    ###########################################################################

    ###########################################################################
    #                          END OF YOUR CODE
    ###########################################################################
    return steps

In [ ]:
# Check: the blocks land where nanograd's gate_bias puts them; memory lengths.
you, lib = MyLSTM(8, 16), ng.LSTM(8, 16)
set_gate_biases(you, i=-5.0, f=5.0); lib.gate_bias(i=-5.0, f=5.0)
assert np.array_equal(you.b, lib.b), 'bias blocks differ from nanograd.LSTM.gate_bias'
for b_f in (0, 1, 3, 5, 7):
    print(f'forget bias {b_f}: memory ~ {memory_steps(b_f):7.1f} steps')
assert abs(memory_steps(0) - 2.0) < 1e-12 and abs(memory_steps(5) - (1 + math.exp(5))) < 1e-9
compare_to_canonical([memory_steps(1), memory_steps(5)], [1 + math.exp(1), 1 + math.exp(5)], labels=['mem(1)', 'mem(5)'])

### Inline Question 4
(a) Show that $1/(1-\sigma(b)) = 1 + e^{b}$. (b) The module's latch fails at $T = 320$ with $b_f = 5$
and $b_i = 0$, and succeeds with $b_f = 5$ and $b_i = -3$. In terms of what accumulates in the cell
during 319 distractors, why? (c) Tallec and Ollivier's (2018) "chrono" initialization draws
$b_f = \log u$ with $u$ uniform on $[1, T_{\max} - 1]$ and sets $b_i = -b_f$. What range of memory lengths
does that give the cells, and why might a spread be better than one value? *Your answer:* ____

## Verify: your pieces reproduce the module's latch

A GIVEN harness trains the latch at $T = 80$ — **your** `MyLSTM` with **your** gate biases, a library
`Linear` read-out, Adam and clipping — with exactly the initial weights and batches of
`canon.train_latch`, for 300 updates. With its gates open to the past and shut to new input your
LSTM must hold the signal through 79 distractors on fresh test sequences; with the textbook biases
(forget 1, input 0) it must fail; trained with the cell-only gradient it must still succeed. All
three must match the canonical runs.

In [ ]:
def train_latch_with(layer_cls, set_bias, bias, cell_only=False, T=80, steps=300, seed=0):
    """GIVEN harness: canon.train_latch with YOUR layer and YOUR bias helper. Do not modify."""
    gen = np.random.default_rng(canon.DATA_SEED + 100 * seed)
    r = Rng(canon.TOY_SEED + 100 * seed)
    lay = layer_cls(canon.V, canon.N_H); ng.xavier_normal(lay.Wx, r); ng.xavier_normal(lay.Wh, r)
    set_bias(lay, **bias)
    head = ng.Linear(canon.N_H, canon.NS); ng.xavier_normal(head.W, r)
    opt, loss_layer, first = ng.Adam(lr=canon.TOY_LR), ng.SoftmaxCrossEntropy(), None
    for k in range(steps):
        S = canon.latch_batch(gen, T, canon.M_TOY)
        H = lay.forward(canon.one_hot(S))
        loss = loss_layer.forward(head.forward(H[-1]), S[0])
        dH = np.zeros_like(H); dH[-1] = head.backward(loss_layer.backward())
        lay.backward(dH, cell_only=cell_only)
        grads = lay.grads() + head.grads()
        if first is None:
            first = (loss, float(sum(g.sum() for g in grads)))
        ng.clip_grad_norm(grads, canon.CLIP)
        opt.step(lay.params() + head.params(), grads)
    return canon.latch_accuracy(lay, head, T, seed=seed), loss, first

lib_bias = lambda lay, i, f: lay.gate_bias(i=i, f=f)
runs = {'open': (canon.OPEN_BIAS, False), 'textbook': (canon.TEXTBOOK_BIAS, False), 'carousel': (canon.OPEN_BIAS, True)}
mine = {k: train_latch_with(MyLSTM, set_gate_biases, b, co) for k, (b, co) in runs.items()}
lib = {k: train_latch_with(ng.LSTM, lib_bias, b, co) for k, (b, co) in runs.items()}
for k in runs:
    print(f'{k:9s} with YOUR pieces: test accuracy {mine[k][0]:.4f}, final batch loss {mine[k][1]:.5f}')
assert mine['open'][0] == 1.0, 'with its gates open to the past and shut to input, the LSTM should hold the signal'
assert mine['textbook'][0] < 0.5, 'with the textbook biases it should not learn this lag'
assert mine['carousel'][0] == 1.0, 'the cell-only gradient should be enough'
canonical = canon.train_latch(T=80, steps=300, bias=canon.OPEN_BIAS)['acc'][80]
compare_to_canonical([mine['open'][2][0], mine['open'][2][1], mine['carousel'][2][1], mine['open'][0], mine['textbook'][0]],
                     [lib['open'][2][0], lib['open'][2][1], lib['carousel'][2][1], canonical, lib['textbook'][0]],
                     labels=['loss0', 'grad0', 'grad0 (cell)', 'acc open', 'acc textbook'])
print('OK — your cell, your backpropagation through it and your gate biases hold a symbol for 80 steps; '
      'the textbook biases cannot; the carousel alone can.')